In [1]:
# 설치
!pip install transformers torch

In [2]:
from google.colab import drive

import pandas as pd
from transformers import pipeline

In [3]:
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/sparta/tp4/review_analysis/data/'
INPUT_PATH = OUTPUT_DIR + 'embedding_input.parquet'

print(f'INPUT     : {INPUT_PATH}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

Mounted at /content/drive
INPUT     : /content/drive/MyDrive/sparta/tp4/review_analysis/data/embedding_input.parquet
OUTPUT_DIR: /content/drive/MyDrive/sparta/tp4/review_analysis/data/


In [4]:
# 데이터 로드
df = pd.read_parquet(INPUT_PATH)
df['리뷰내용_clean'] = df['리뷰내용_clean'].fillna('').astype(str)

print(f'로드 완료: {len(df):,}건')
print(f'평점 분포:\n{df["평점"].value_counts().sort_index()}')

로드 완료: 598,301건
평점 분포:
평점
1.0      1587
2.0      2013
3.0     15953
4.0     85350
5.0    491770
Name: count, dtype: int64


---

# 모델 비교

In [5]:
# 샘플 추출 (평점 분포 맞춰서)
# 평점별 20건씩 → 총 100건
sample = pd.concat([
    df[df['평점'] <= 2].sample(20, random_state=42),   # 부정 후보
    df[df['평점'] == 3].sample(20, random_state=42),   # 중립
    df[df['평점'] >= 4].sample(60, random_state=42),   # 긍정 후보
]).reset_index(drop=True)

print(f'샘플 수: {len(sample)}건')
print(f'평점 분포:\n{sample["평점"].value_counts().sort_index()}')

샘플 수: 100건
평점 분포:
평점
1.0     7
2.0    13
3.0    20
4.0    10
5.0    50
Name: count, dtype: int64


In [6]:
# 공통 추론 함수
def get_results(model, texts):
    results = []
    for text in texts:
        try:
            result = model(str(text)[:512])[0]
            results.append({'label': result['label'], 'score': round(result['score'], 4)})
        except:
            results.append({'label': 'ERROR', 'score': 0.0})
    return results

In [7]:
# 모델 1: KR-FinBert
# 금융 도메인 기반 한국어 감성 분석 모델
finbert = pipeline('text-classification', model='snunlp/KR-FinBert-SC', device=0)

res = get_results(finbert, sample['리뷰내용_clean'])
sample['finbert_label'] = [r['label'] for r in res]
sample['finbert_score'] = [r['score'] for r in res]
print('FinBert 완료')
print(sample['finbert_label'].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/406M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/406M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

FinBert 완료
finbert_label
neutral    100
Name: count, dtype: int64


In [8]:
# 모델 2: KoELECTRA
# 한국어 감성 분석 파인튜닝 모델
koelectra = pipeline('text-classification', model='monologg/koelectra-base-finetuned-sentiment', device=0)

res = get_results(koelectra, sample['리뷰내용_clean'])
sample['koelectra_label'] = [r['label'] for r in res]
sample['koelectra_score'] = [r['score'] for r in res]
print('KoELECTRA 완료')
print(sample['koelectra_label'].value_counts())

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/441M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/441M [00:00<?, ?B/s]

ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-finetuned-sentiment
Key                        | Status     | 
---------------------------+------------+-
classifier.weight          | UNEXPECTED | 
classifier.bias            | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

KoELECTRA 완료
koelectra_label
positive    89
negative    11
Name: count, dtype: int64


In [9]:
# 모델 3: KlueBERT
# KLUE 기반 한국어 감성 분석 모델
kluebert = pipeline('text-classification', model='hun3359/klue-bert-base-sentiment', device=0)

res = get_results(kluebert, sample['리뷰내용_clean'])
sample['kluebert_label'] = [r['label'] for r in res]
sample['kluebert_score'] = [r['score'] for r in res]
print('KlueBERT 완료')
print(sample['kluebert_label'].value_counts())

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: hun3359/klue-bert-base-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

KlueBERT 완료
kluebert_label
만족스러운          60
남의 시선을 의식하는    10
흥분              5
느긋              3
조심스러운           3
부끄러운            3
안달하는            2
툴툴대는            2
신이 난            2
한심한             1
혐오스러운           1
실망한             1
방어적인            1
스트레스 받는         1
짜증내는            1
가난한 불우한         1
기쁨              1
편안한             1
성가신             1
Name: count, dtype: int64


In [10]:
# 전체 비교 출력
print('\n=== 모델 비교 ===\n')
print(f"{'평점':>3} | {'평점기반':>5} | {'FinBert':>8} | {'KoELECTRA':>10} | {'KlueBert':>10} | 리뷰")
print('-' * 130)

for _, row in sample.iterrows():
    rating_label = '긍정' if row['평점'] >= 4 else ('부정' if row['평점'] <= 2 else '중립')
    print(f"{row['평점']:>3} | {rating_label:>5} | {row['finbert_label']:>6}({row['finbert_score']:.2f}) | {row['koelectra_label']:>8}({row['koelectra_score']:.2f}) | {row['kluebert_label']:>8}({row['kluebert_score']:.2f}) | {str(row['리뷰내용_clean'])[:40]}")


=== 모델 비교 ===

 평점 |  평점기반 |  FinBert |  KoELECTRA |   KlueBert | 리뷰
----------------------------------------------------------------------------------------------------------------------------------
2.0 |    부정 | neutral(1.00) | positive(0.51) |    만족스러운(0.39) | 182.9 73 인데 라지가 사이즈가 적당 합니다 색깔도 좋아요
1.0 |    부정 | neutral(1.00) | positive(0.51) |     안달하는(0.11) | 주문 4일지나도 배송확인 안돼서 정확한 상황이랑 수령일자 1:1문의했더니
1.0 |    부정 | neutral(1.00) | positive(0.50) |    만족스러운(0.46) | 꾀 쓸만하다 적당한 오버핏에 좋나 색감도맘에듬
2.0 |    부정 | neutral(0.99) | positive(0.50) | 남의 시선을 의식하는(0.37) | 커도 너무 큽니다 두사이즈 다운해야 맞을듯 너무 옛날 힙합 카고 느낌..
2.0 |    부정 | neutral(0.87) | positive(0.51) |    조심스러운(0.19) | 생각했던 재질 보다 더 바스락 바스락 거려서 못입겠어요..
1.0 |    부정 | neutral(1.00) | negative(0.50) |      한심한(0.29) | 가을에 입으려고 9월에 시켰는데 이제야 배송이 왔어요 ^^ 9월 주문도 
1.0 |    부정 | neutral(1.00) | positive(0.51) | 남의 시선을 의식하는(0.62) | 빨자마자 목 주름 엄청 생겼어요 바로 잠옷행입니다
2.0 |    부정 | neutral(1.00) | positive(0.51) |    혐오스러운(0.11) | 목파임이 생각보다 심해서 늘어난 아저씨 러닝같음
2.0 |   

In [11]:
# 부정 리뷰(평점 1~2)에서 모델별 정확도 확인
print('\n=== 부정 리뷰(평점 1~2) 모델별 비교 ===\n')

neg_sample = sample[sample['평점'] <= 2]
for _, row in neg_sample.iterrows():
    print(f"평점={row['평점']}")
    print(f"  FinBert:   {row['finbert_label']} ({row['finbert_score']:.2f})")
    print(f"  KoELECTRA: {row['koelectra_label']} ({row['koelectra_score']:.2f})")
    print(f"  KlueBert:  {row['kluebert_label']} ({row['kluebert_score']:.2f})")
    print(f"  리뷰: {str(row['리뷰내용_clean'])[:100]}")
    print()


=== 부정 리뷰(평점 1~2) 모델별 비교 ===

평점=2.0
  FinBert:   neutral (1.00)
  KoELECTRA: positive (0.51)
  KlueBert:  만족스러운 (0.39)
  리뷰: 182.9 73 인데 라지가 사이즈가 적당 합니다 색깔도 좋아요

평점=1.0
  FinBert:   neutral (1.00)
  KoELECTRA: positive (0.51)
  KlueBert:  안달하는 (0.11)
  리뷰: 주문 4일지나도 배송확인 안돼서 정확한 상황이랑 수령일자 1:1문의했더니, 회신도 없이 뭉개고 문의한 이후에 그제서야 발송해서 주문하고 1주일만에 수령. 급한분들은 다른상품 주문하

평점=1.0
  FinBert:   neutral (1.00)
  KoELECTRA: positive (0.50)
  KlueBert:  만족스러운 (0.46)
  리뷰: 꾀 쓸만하다 적당한 오버핏에 좋나 색감도맘에듬

평점=2.0
  FinBert:   neutral (0.99)
  KoELECTRA: positive (0.50)
  KlueBert:  남의 시선을 의식하는 (0.37)
  리뷰: 커도 너무 큽니다 두사이즈 다운해야 맞을듯 너무 옛날 힙합 카고 느낌...

평점=2.0
  FinBert:   neutral (0.87)
  KoELECTRA: positive (0.51)
  KlueBert:  조심스러운 (0.19)
  리뷰: 생각했던 재질 보다 더 바스락 바스락 거려서 못입겠어요..

평점=1.0
  FinBert:   neutral (1.00)
  KoELECTRA: negative (0.50)
  KlueBert:  한심한 (0.29)
  리뷰: 가을에 입으려고 9월에 시켰는데 이제야 배송이 왔어요 ^^ 9월 주문도 제대로 처리안해놓고선 아무렇지 않게 계속 상품 올려놓고 판매하는게 웃겨요

평점=1.0
  FinBert:   neutral (1.00)
  KoELECTRA: positive (0.51)
 

### 모델별 평가
- FinBert 문제
  - 거의 모든 리뷰를 `neutral`로 분류
  - 금융 도메인 모델이라 의류 도메인과 맞지 않음

- KoELECTRA 문제
  - 긍정/부정은 어느 정도 구분하나 점수가 항상 0.50~0.54로 너무 낮음
  - 부정 리뷰도 `positive`로 잡고 있어 신뢰도 낮음

- KlueBERT
  - 감성 분석 목적과 다른 모델

- 평점 기반 + 규칙 보완으로 감성 리뷰를 진행해야할듯

---

# 평점 기반

## 평점 4~5점 중 부정 리뷰 비율 확인

In [12]:
# 평점 4~5인데 부정 키워드 있는 리뷰 비율 확인
negative_keywords = ['별로', '실망', '불편', '아쉽', '환불', '반품', '교환', '최악', '후회', '못입']

high_rating = df[df['평점'] >= 4].copy()
high_rating['has_negative'] = high_rating['리뷰내용_clean'].apply(
    lambda x: any(kw in str(x) for kw in negative_keywords)
)

total = len(high_rating)
negative_count = high_rating['has_negative'].sum()

print(f'평점 4~5 전체: {total:,}건')
print(f'부정 키워드 포함: {negative_count:,}건 ({negative_count/total*100:.1f}%)')
print()

# 평점 5인데 부정 키워드 있는 샘플 확인
print('=== 평점 5인데 부정 키워드 있는 샘플 ===')
for _, row in high_rating[(high_rating['평점'] == 5) & (high_rating['has_negative'])].sample(5, random_state=42).iterrows():
    print(f"  평점={row['평점']} | {str(row['리뷰내용_clean'])[:80]}")

평점 4~5 전체: 577,120건
부정 키워드 포함: 24,223건 (4.2%)

=== 평점 5인데 부정 키워드 있는 샘플 ===
  평점=5.0 | 이런 옷은 처음 입어보는데 불편하지도 않고 입기 좋아요
  평점=5.0 | 입을만한데 색깔이 조금 별로라서 색 잘 보고 사셔야할듯
  평점=5.0 | 158/48인데 이거 자체가 엄청 오버핏은 아닌듯한 옷인거같아용 오버핏이긴한디 뭔가 더 저는 왕큰핏을 기대했는데 아쉽 솔까 L시켜도 될뻔했음 후
  평점=5.0 | 아주 좋습니데이 편하게 입을 수 있어요 기모가 있어서 아쉽지만 그래서 더 탄탄한 느낌도 드네요 가성비로 입어요!
  평점=5.0 | 기모로 잘 못 시켜서 교환했어요 같은 사이즈인데도 기모후드보다 작은거 같습니다


In [13]:
# 평점 3점 리뷰 100건 샘플 추출
sample_3 = df[df['평점'] == 3].sample(100, random_state=42).reset_index(drop=True)

print(f'평점 3점 샘플: {len(sample_3)}건')

평점 3점 샘플: 100건


In [14]:
# KoELECTRA로 평점 3점 리뷰 분류
koelectra = pipeline(
    'text-classification',
    model='monologg/koelectra-base-finetuned-sentiment',
    device=0
)

res = get_results(koelectra, sample_3['리뷰내용_clean'])
sample_3['koelectra_label'] = [r['label'] for r in res]
sample_3['koelectra_score'] = [r['score'] for r in res]

print('KoELECTRA 완료')
print(f'\n분류 결과:')
print(sample_3['koelectra_label'].value_counts())

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-finetuned-sentiment
Key                        | Status     | 
---------------------------+------------+-
classifier.weight          | UNEXPECTED | 
classifier.bias            | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


KoELECTRA 완료

분류 결과:
koelectra_label
negative    71
positive    29
Name: count, dtype: int64


In [15]:
# 결과 확인
print('\n=== 평점 3점 KoELECTRA 분류 결과 ===\n')

for _, row in sample_3.iterrows():
    print(f"  {row['koelectra_label']:>8}({row['koelectra_score']:.2f}) | {str(row['리뷰내용_clean'])[:80]}")


=== 평점 3점 KoELECTRA 분류 결과 ===

  negative(0.55) | 오버핏 조아해서 엑스라지했는데 마대자루핏 완성임 라지사세요 키160 옷 자체는 튼튼하고 막입기 좋음 갸꿀
  negative(0.53) | 통이 좀만 줄었으면 좋겠네요 대신 엄청 편하게 입을 수 있어요
  negative(0.55) | 좋아요 ㅎㅎ 세탁 후 변형 없습니다 배송도 괜찮고 색감도 같아요
  negative(0.56) | 빈티지한게 아주 좋고요 흔하다고 들었는데 아직못봤어요
  negative(0.55) | 진짜 스탠다드핏으로 최고임... 다른색도 살거임.. ㄹㅇ 개좋음요
  negative(0.55) | 핏이 너무 맘에 들고 포인트도 있어서 좋아요 스트랩도 있어서 편해요
  negative(0.55) | 여자 키 169이고 상의 55사이즈 입어요 M사이즈 샀는데 어깨부분이 굉ㅇ장히 남고 큽니다 그래도 예뻐서 자주 입습니당
  negative(0.54) | 색감이 이쁘고 재질 부드러워요 사이즈 정 사이즈가면 오버하게 입을수 있을거 같아요
  positive(0.52) | 완전 뻣뻣한 비닐..? 얇은 호루 ...? 재질이 낯설고 좀 웃겨요 막 입을만은 한데 소리가 많이 거슬리네요
  negative(0.54) | 옷감도 괜찮고 좋네요. 제가 상체가 꽤 큰편이라 2xl를 했는데 좀 크네요. 그냥 xl했어도 될 듯~ 사이즈 선택 잘 참고하세여~
  negative(0.55) | 원단이 제법 두툼해서 아주 더운날 입기는 힘들겠네요. 총장이 짧아서 레이어드 필수입니다
  negative(0.51) | 재봉상태가 좋은편은 아닙니다 세탁한번에 이렇게 올들이 풀리는건 ㅋㅋ
  negative(0.53) | 작은사이즈도 있었음 좋겠어요. 체형작은분께는 s도 크네요. 참고하세요~
  positive(0.50) | 재질이 빳빳할 줄 알았는데 먼지 잘 붙는 재질이에요ㅠ 목부분도 약간 각져있어서 이상해요 디자인은 예뻐요
  negative(0.50) | 모자가 크고 따뜻함...모자가

In [16]:
# 점수 분포 확인
print('\n=== 점수 분포 ===')
print(sample_3.groupby('koelectra_label')['koelectra_score'].describe())


=== 점수 분포 ===
                 count      mean       std     min     25%     50%     75%  \
koelectra_label                                                              
negative          71.0  0.537137  0.015519  0.5016  0.5282  0.5420  0.5497   
positive          29.0  0.511079  0.007740  0.5007  0.5052  0.5084  0.5155   

                    max  
koelectra_label          
negative         0.5571  
positive         0.5271  
